In [1]:
!pip install crewai
!pip install crewai-tools
!pip install databricks-sdk

  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached chromadb-1.1.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.2 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached portalocker-2.7.0-py2.py3-none-any.whl.metadata (6.8 kB)
  Using cached pydantic_settings-2.10.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.37.0-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.4/814.4 kB 19.4 MB/s  0:00:00
Usin

In [6]:
import os
import getpass
def set_if_undefined(var:str):
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
set_if_undefined("SERPER_API_KEY")
set_if_undefined("GROQ_API_KEY")

GROQ_API_KEY ········


In [3]:
from crewai_tools import SerperDevTool

In [4]:
search_tool = SerperDevTool()
print(type(search_tool))

<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool'>


In [5]:
search_query = "Latest breakthroughs in agentic ai"
search_results = search_tool.run(query = search_query)
print(f"The search results are :{search_results}")

The search results are :{'searchParameters': {'q': 'Latest breakthroughs in agentic ai', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Agentic AI advances | McKinsey & Company', 'link': 'https://www.mckinsey.com/featured-insights/week-in-charts/agentic-ai-advances', 'snippet': 'January 8, 2026 AI is becoming widely used, but only a minority of companies are scaling more sophisticated capabilities, such as agents, into ...', 'position': 1}, {'title': '10+ Agentic AI Trends and Examples for 2026 - AIMultiple', 'link': 'https://aimultiple.com/agentic-ai-trends', 'snippet': '10+ Agentic AI Trends and Examples for 2026 · 1. Towards autonomous, self-healing data pipelines · 2. Tooling over process · 3. Vertical AI agents in ...', 'position': 2, 'sitelinks': [{'title': '10+ agentic AI trends and...', 'link': 'https://aimultiple.com/agentic-ai-trends#10-agentic-ai-trends-and-examples'}, {'title': 'Tooling over process', 'link': 'https://aimultiple.com/agentic-ai-tren

In [7]:
from crewai import LLM
llm = LLM(
    model = "groq/llama-3.3-70b-versatile",
    api_key = os.environ["GROQ_API_KEY"],
    max_tokens = 2000,
    temperature = 0.6
)


In [8]:
from crewai import Agent
research_agent = Agent(
   role = "Senior research analyst",
   goal='Uncover cutting-edge information and insights on any subject with comprehensive analysis',
   backstory = """You are an expert researcher with extensive experience in gathering, analyzing, and synthesizing information across multiple domains. 
   Your analytical skills allow you to quickly identify key trends, separate fact from opinion, and produce insightful reports on any topic. 
   You excel at finding reliable sources and extracting valuable information efficiently.""",
   verbose = True,
   allow_delegation = False,
   llm = llm,
   tools = [SerperDevTool()] 
)

In [9]:
research_agent

Agent(role=Senior research analyst, goal=Uncover cutting-edge information and insights on any subject with comprehensive analysis, backstory=You are an expert researcher with extensive experience in gathering, analyzing, and synthesizing information across multiple domains. 
   Your analytical skills allow you to quickly identify key trends, separate fact from opinion, and produce insightful reports on any topic. 
   You excel at finding reliable sources and extracting valuable information efficiently.)

In [12]:
writer_agent = Agent(
    role = "Tech content Strategist",
    goal = 'Craft well-structured and engaging content based on research findings',
    backstory = """You are a skilled content strategist known for translating 
    complex topics into clear and compelling narratives. Your writing makes 
    information accessible and engaging for a wide audience.""",
    verbose = True,
    allow_delegation = False,
    llm = llm,
    
)

In [13]:
writer_agent

Agent(role=Tech content Strategist, goal=Craft well-structured and engaging content based on research findings, backstory=You are a skilled content strategist known for translating 
    complex topics into clear and compelling narratives. Your writing makes 
    information accessible and engaging for a wide audience.)

In [14]:
from crewai import Task
resarch_task = Task(
   description = "Analyze the major {topic},identifying key trends and technologies.Provide a detailed report on their potential impact",
   agent = research_agent,
   expected_output = "A detailed report on {topic} ,including trends ,emerging technologies and their impact" 
)

In [15]:
writer_task = Task(
    description = "Create an engaging blog post based on research findings about {topic}.Tailor the content for tech-savy audience ensuring clarity and intrest.",
    agent = writer_agent,
    expected_output = "A 4 paragraph post on {topic},written clearly and engaging for tech enthusiasts"
    
)

In [16]:
from crewai import Crew, Process
crew = Crew(
    tasks = [resarch_task, writer_task],
    agents = [research_agent,writer_agent],
    verbose = True,
    process = Process.sequential
)

In [17]:
result = crew.kickoff(inputs = {"topic": "Latest trends in Agentic Ai"})


╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name:                                                                       │
│  crew                                                                        │
│  ID:                                                                         │
│  0c397444-8c84-435c-8bf2-d7af2671b346                                        │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started              

In [18]:
final_output = result.raw
print(f"Final output is {final_output}")

Final output is The latest trends in Agentic Ai are revolutionizing the way organizations operate and interact with their customers. With the agentic AI market expected to reach $45 billion by 2030, up from $8.5 billion in 2026, it's clear that this technology is poised for significant growth and innovation. One of the key trends in Agentic Ai is the rise of hyperautomation, which enables organizations to automate complex processes and improve efficiency. Additionally, AI-powered decision intelligence is becoming increasingly important, as it allows organizations to make data-driven decisions and drive business success. Personalized employee experiences and enhanced customer interactions are also key trends, as they enable organizations to improve employee engagement and customer satisfaction.

Another significant trend in Agentic Ai is the move towards autonomous, self-healing data pipelines. This allows organizations to improve the accuracy and reliability of their data, which is cri

In [20]:
tasks_outputs = result.tasks_output
tasks_outputs

[TaskOutput(description='Analyze the major Latest trends in Agentic Ai,identifying key trends and technologies.Provide a detailed report on their potential impact', name='Analyze the major Latest trends in Agentic Ai,identifying key trends and technologies.Provide a detailed report on their potential impact', expected_output='A detailed report on Latest trends in Agentic Ai ,including trends ,emerging technologies and their impact', summary='Analyze the major Latest trends in Agentic Ai,identifying key trends...', raw='The latest trends in Agentic Ai include the rise of hyperautomation, AI-powered decision intelligence, personalized employee experiences, enhanced customer interactions, and AI for improved forecasting. Agentic AI is poised to reach $45 billion by 2030, up from $8.5 billion in 2026, as organizations look to integrate agentic systems into their operations. \n\nKey trends in Agentic Ai include:\n1. Towards autonomous, self-healing data pipelines\n2. Tooling over process\n3

In [21]:
print("Task Description", tasks_outputs[0].description)
print("Output of research task ",tasks_outputs[0])

Task Description Analyze the major Latest trends in Agentic Ai,identifying key trends and technologies.Provide a detailed report on their potential impact
Output of research task  The latest trends in Agentic Ai include the rise of hyperautomation, AI-powered decision intelligence, personalized employee experiences, enhanced customer interactions, and AI for improved forecasting. Agentic AI is poised to reach $45 billion by 2030, up from $8.5 billion in 2026, as organizations look to integrate agentic systems into their operations. 

Key trends in Agentic Ai include:
1. Towards autonomous, self-healing data pipelines
2. Tooling over process
3. Vertical AI agents in specialized industries
4. Integration of AI agents with other technologies
5. Agentic AI reshaping team roles

The most recent step in the evolution of Agentic Ai was human/machine conversation. In 2025, agentic AI changed how a large swath of developers write code. 

Agentic Ai has the potential to revolutionize various ind

In [22]:
print("Writer task description:", tasks_outputs[1].description)
print(" \nOutput of writer task:", tasks_outputs[1].raw)

Writer task description: Create an engaging blog post based on research findings about Latest trends in Agentic Ai.Tailor the content for tech-savy audience ensuring clarity and intrest.
 
Output of writer task: The latest trends in Agentic Ai are revolutionizing the way organizations operate and interact with their customers. With the agentic AI market expected to reach $45 billion by 2030, up from $8.5 billion in 2026, it's clear that this technology is poised for significant growth and innovation. One of the key trends in Agentic Ai is the rise of hyperautomation, which enables organizations to automate complex processes and improve efficiency. Additionally, AI-powered decision intelligence is becoming increasingly important, as it allows organizations to make data-driven decisions and drive business success. Personalized employee experiences and enhanced customer interactions are also key trends, as they enable organizations to improve employee engagement and customer satisfaction.

In [23]:
print("We can get the agent for researcher task:  ",tasks_outputs[0].agent)
print("We can get the agent for the writer task: ",tasks_outputs[1].agent)

We can get the agent for researcher task:   Senior research analyst
We can get the agent for the writer task:  Tech content Strategist


In [24]:
token_count = result.token_usage.total_tokens
prompt_tokens = result.token_usage.prompt_tokens
completion_tokens = result.token_usage.completion_tokens

print(f"Total tokens used: {token_count}")
print(f"Prompt tokens: {prompt_tokens} (used for instructions to the model)")
print(f"Completion tokens: {completion_tokens} (generated in response)")

Total tokens used: 13396
Prompt tokens: 8136 (used for instructions to the model)
Completion tokens: 5260 (generated in response)
